# Batch Jitter Correction (standalone)

**Jitter-only** pipeline: run after synchronization is done. Assumes each block already has an analysis folder (from `batch_block_synchronization.ipynb`) with the same `analysis_subfolder_name`.

1. **Block setup** — Create block collection and set `analysis_path` to the existing analysis subfolder.
2. **ROI selection** — For each block, load jitter ROIs from disk if saved (`jitter_rois.pkl` in the block's analysis folder); otherwise run manual selection and **save** ROIs so the next run loads them automatically.
3. **Jitter computation** — Run cross-correlation jitter reports in parallel. Blocks that already have `jitter_report_dict.pkl` are **skipped** unless `overwrite_jitter_reports=True`. Progress bars and clear status messages (OK / FAILED) are printed.

No LED blink detection or plots in this notebook. Run locally to debug, then use the same notebook on the remote machine with a larger dataset.

## 1. Imports

In [1]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import pickle
import numpy as np
import pandas as pd
import cv2
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import Manager
import threading
try:
    from tqdm.notebook import tqdm as tqdm_notebook
except ImportError:
    from tqdm import tqdm as tqdm_notebook

from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing.block_sync_core import run_jitter_report_worker

JITTER_ROIS_FILENAME = "jitter_rois.pkl"

## 2. Configuration

In [2]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")
block_numbers = [[6],[15]]
animal = ["PV_126","PV_106"]
bad_blocks = []

# Must match the analysis subfolder used in batch_block_synchronization (e.g. batch_analysis_output_2026_02_05)
analysis_subfolder_name = "batch_analysis_output_" + datetime.now().strftime("%Y_%m_%d")
# Or set explicitly: analysis_subfolder_name = "batch_analysis_output_2026_02_05"

overwrite_jitter_reports = False  # Set True to recompute even when jitter_report_dict.pkl exists
max_workers = 2  # Limit parallelism to avoid OOM; increase (e.g. 4) if you have enough RAM
jitter_verbose = True  # Show one tqdm progress bar per block

## 3. Create block collection and set analysis paths

In [3]:
def create_run_folder_for_block(block, analysis_subfolder_name):
    base = block.block_path / "analysis"
    base.mkdir(parents=True, exist_ok=True)
    run_path = base / analysis_subfolder_name
    run_path.mkdir(parents=True, exist_ok=True)
    block.analysis_path = run_path
    return run_path

block_collection = list(uf.block_generator(
    block_numbers=block_numbers,
    experiment_path=experiment_path,
    animal=animal,
    bad_blocks=bad_blocks,
))
for b in block_collection:
    create_run_folder_for_block(b, analysis_subfolder_name)

print(f"Blocks: {[b.block_num for b in block_collection]}")
print(f"Analysis subfolder: {analysis_subfolder_name}")
print(f"Example path: {block_collection[0].analysis_path}")

instantiated block number 006 at Path: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006, new OE version
Found the sample rate for block 006 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_126\2024_07_18\block_006\oe_files\PV126_Trial15_hunter7_2024-07-18_12-25-35\Record Node 102...
xml data matches file data.

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode)
retrieving zertoh sample number for block 006
got it!
instantiated block number 015 at Path: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015, new OE version
Found the sample rate for block 015 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\oe_files\PV106_IMU_trial4_prey_2025-09-04_13-24-17\

## 4. Jitter ROI selection (saved to each block's analysis folder)

In [4]:
def _ensure_odd_roi(roi):
    x, y, w, h = roi
    if w % 2 == 0:
        w += 1
    if h % 2 == 0:
        h += 1
    return [x, y, w, h]

jitter_rois_by_block = {}
for block in block_collection:
    block.handle_eye_videos()
    if not block.le_videos or not block.re_videos:
        print(f"[Block {block.block_num}] SKIP: no eye videos.")
        continue
    roi_path = block.analysis_path / JITTER_ROIS_FILENAME
    if roi_path.exists():
        with open(roi_path, "rb") as f:
            rois = pickle.load(f)
        jitter_rois_by_block[block.block_num] = rois
        print(f"[Block {block.block_num}] Loaded ROIs from {roi_path.name}")
    else:
        rois = {}
        for eye_label, vid_path in [("Left eye (jitter)", block.le_videos[0]), ("Right eye (jitter)", block.re_videos[0])]:
            cap = cv2.VideoCapture(str(vid_path))
            if not cap.isOpened():
                raise RuntimeError(f"Cannot open {vid_path}")
            ret, frame = cap.read()
            cap.release()
            if not ret:
                raise RuntimeError(f"Cannot read first frame: {vid_path}")
            roi = list(cv2.selectROI(f"Block {block.block_num} - {eye_label}", frame, showCrosshair=True, fromCenter=False))
            cv2.destroyWindow(f"Block {block.block_num} - {eye_label}")
            roi = _ensure_odd_roi(roi)
            key = "left_roi" if "Left" in eye_label else "right_roi"
            rois[key] = roi
        with open(roi_path, "wb") as f:
            pickle.dump(rois, f)
        jitter_rois_by_block[block.block_num] = rois
        print(f"[Block {block.block_num}] Selected ROIs and saved to {roi_path.name}")

print(f"\nReady: {len(jitter_rois_by_block)} block(s) have jitter ROIs.")

handling eye video files
converting videos...
h264 files found: 2; already have .mp4 (skip): 2; to convert: 0
no eye videos to convert (all .mp4 present or no .h264); continuing to validate and set video lists.
Validating videos...
The video named hunter7_LE.mp4 has reported 79224 frames and has 79224 frames, it has dropped 0 frames
The video named hunter7.mp4 has reported 79217 frames and has 79217 frames, it has dropped 0 frames
[Block 006] Selected ROIs and saved to jitter_rois.pkl
handling eye video files
converting videos...
h264 files found: 2; already have .mp4 (skip): 2; to convert: 0
no eye videos to convert (all .mp4 present or no .h264); continuing to validate and set video lists.
Validating videos...
The video named imu_trial4_prey_LE.mp4 has reported 19800 frames and has 19800 frames, it has dropped 0 frames
The video named imu_trial4_prey.mp4 has reported 19701 frames and has 19701 frames, it has dropped 0 frames
[Block 015] Selected ROIs and saved to jitter_rois.pkl

Rea

## 5. Run jitter computation (parallel, skip if report exists)

In [ ]:
blocks_with_rois = [b for b in block_collection if b.block_num in jitter_rois_by_block]
if not blocks_with_rois:
    print("No blocks have jitter ROIs. Run the ROI cell above first.")
else:
    worker_args = []
    for block in blocks_with_rois:
        bn = block.block_num
        roi_dict = jitter_rois_by_block[bn]
        base_args = (
            str(block.le_videos[0]),
            str(block.re_videos[0]),
            roi_dict["left_roi"],
            roi_dict["right_roi"],
            str(block.analysis_path),
            overwrite_jitter_reports,
        )
        worker_args.append((base_args, bn))

    n_workers = min(len(worker_args), max_workers or len(worker_args))
    print(f"Jitter computation: {len(worker_args)} block(s), max_workers={n_workers}. Existing reports skipped unless overwrite_jitter_reports=True.\n")
    jitter_results = []
    jitter_failures = []
    block_by_path = {str(b.analysis_path): b for b in blocks_with_rois}

    listener = None
    if jitter_verbose:
        manager = Manager()
        progress_queue = manager.Queue()
        full_args = [a[0] + (progress_queue, a[1]) for a in worker_args]
        bars = {}
        done_count = [0]
        num_blocks = len(full_args)

        def progress_listener():
            while done_count[0] < num_blocks:
                try:
                    msg = progress_queue.get(timeout=0.5)
                except Exception:
                    continue
                if msg[0] == "init":
                    _, block_id, total = msg
                    bars[block_id] = tqdm_notebook(total=total, desc=f"Block {block_id}", unit="frame")
                elif msg[0] == "progress":
                    _, block_id, current, total = msg
                    if block_id in bars:
                        bars[block_id].n = current
                        bars[block_id].refresh()
                elif msg[0] == "done":
                    _, block_id = msg
                    if block_id in bars:
                        bars[block_id].close()
                    done_count[0] += 1
            return None

        listener = threading.Thread(target=progress_listener, daemon=True)
        listener.start()
    else:
        full_args = [a[0] for a in worker_args]

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(run_jitter_report_worker, a): a for a in full_args}
        for future in as_completed(futures):
            args = futures[future]
            ap = args[4]
            block = block_by_path.get(ap)
            bn = block.block_num if block else "?"
            try:
                status, path_str, err = future.result()
                if status == "skipped":
                    print(f"  Block {bn}: OK (skipped — already has jitter report)")
                elif status == "computed":
                    print(f"  Block {bn}: OK (computed and saved)")
                else:
                    print(f"  Block {bn}: FAILED — {err}")
                    jitter_failures.append((block, err or "unknown"))
                    continue
                if block is not None:
                    jitter_results.append(block)
            except Exception as e:
                print(f"  Block {bn}: FAILED — {e}")
                jitter_failures.append((block, str(e)))

    if listener is not None and listener.is_alive():
        listener.join(timeout=5.0)

    # Load jitter dicts into blocks (no LED blink removal, no plots)
    for block in jitter_results:
        block.get_jitter_reports(overwrite=False, sort_on_loading=True, remove_led_blinks=False)

    print(f"\nDone. Jitter reports: {len(jitter_results)} ok, {len(jitter_failures)} failed.")
    if jitter_failures:
        for block, err in jitter_failures:
            print(f"  Block {block.block_num if block else '?'}: {err}")